# Hourly Rental Aggregation and Time Feratures
The goal of this notebook is to transform raw bike rental records into an hourly demand dataset, and add hourly weather data.

The notebook is struectured as follew:

1. Convert rental timestamps to datetime
2. Floor timestamps to the hour
3. Aggregate rental events by hour
4. Combine registered and direct rental counts
5. Create time-based features
6. Merge with weather data

## 1. Load and convert datetime

In [67]:
# this has been done in the first part
import pandas as pd

registered = pd.read_csv("../data/registered_bike_rentals.csv")
direct = pd.read_csv("../data/direct_pickup_bike_rentals.csv")
registered["datetime"] = pd.to_datetime(registered["datetime"])
direct["datetime"] = pd.to_datetime(direct["datetime"])

## 2. Round datetime to hour

In [68]:
# each timestamps is floored to the beginning of its hour
registered["hour"] = registered["datetime"].dt.floor("h")
direct["hour"] = direct["datetime"].dt.floor("h")

## 3. Group rentals by hour

In [69]:
registered_hourly = (
    registered
    .groupby("hour")
    .size()
    .reset_index(name="registered_count")
)

registered_hourly.head()

,hour,registered_count
0,2011-01-01 00:00:00,13
1,2011-01-01 01:00:00,32
2,2011-01-01 02:00:00,27
3,2011-01-01 03:00:00,10
4,2011-01-01 04:00:00,1


In [70]:
direct_hourly = (
    direct
    .groupby("hour")
    .size()
    .reset_index(name="direct_count")
)

direct_hourly.head()

,hour,direct_count
0,2011-01-01 00:00:00,3
1,2011-01-01 01:00:00,8
2,2011-01-01 02:00:00,5
3,2011-01-01 03:00:00,3
4,2011-01-01 06:00:00,2


## 4. Combine registered and direct hourly counts

In [71]:
# outer merge to show missing values
hourly_rentals = registered_hourly.merge(
    direct_hourly,
    on="hour",
    how="outer"
)

hourly_rentals.head()

,hour,registered_count,direct_count
0,2011-01-01 00:00:00,13.0,3.0
1,2011-01-01 01:00:00,32.0,8.0
2,2011-01-01 02:00:00,27.0,5.0
3,2011-01-01 03:00:00,10.0,3.0
4,2011-01-01 04:00:00,1.0,NaN


It shows there are missing counts in one rental type. So we have to replace them with 0 to avoid unkown data:

In [72]:
hourly_rentals[["registered_count", "direct_count"]] = (
    hourly_rentals[["registered_count", "direct_count"]].fillna(0)
)

hourly_rentals[["registered_count", "direct_count"]] = (
    hourly_rentals[["registered_count", "direct_count"]].astype(int)
)

hourly_rentals.head()

,hour,registered_count,direct_count
0,2011-01-01 00:00:00,13,3
1,2011-01-01 01:00:00,32,8
2,2011-01-01 02:00:00,27,5
3,2011-01-01 03:00:00,10,3
4,2011-01-01 04:00:00,1,0


Then we create the total rental count:

In [73]:
hourly_rentals["total_count"] = (
    hourly_rentals["registered_count"] + hourly_rentals["direct_count"]
)

hourly_rentals.head()

,hour,registered_count,direct_count,total_count
0,2011-01-01 00:00:00,13,3,16
1,2011-01-01 01:00:00,32,8,40
2,2011-01-01 02:00:00,27,5,32
3,2011-01-01 03:00:00,10,3,13
4,2011-01-01 04:00:00,1,0,1


In [74]:
# in case data is hourly data is not ordered by time
hourly_rentals = hourly_rentals.sort_values("hour").reset_index(drop=True)
hourly_rentals.head()

,hour,registered_count,direct_count,total_count
0,2011-01-01 00:00:00,13,3,16
1,2011-01-01 01:00:00,32,8,40
2,2011-01-01 02:00:00,27,5,32
3,2011-01-01 03:00:00,10,3,13
4,2011-01-01 04:00:00,1,0,1


## 5. Derive time-based features

In [75]:
hourly_rentals["date"] = hourly_rentals["hour"].dt.date  # for holidays merging
hourly_rentals["hour_of_day"] = hourly_rentals["hour"].dt.hour # demand related: rush hour, lunch time..
hourly_rentals["day_of_week"] = hourly_rentals["hour"].dt.dayofweek # weekdays and weekends
hourly_rentals["month"] = hourly_rentals["hour"].dt.month # seasonal demand
hourly_rentals["is_weekend"] = hourly_rentals["day_of_week"].isin([5, 6]).astype(int)

hourly_rentals.head()

,hour,registered_count,direct_count,total_count,date,hour_of_day,day_of_week,month,is_weekend
0,2011-01-01 00:00:00,13,3,16,2011-01-01,0,5,1,1
1,2011-01-01 01:00:00,32,8,40,2011-01-01,1,5,1,1
2,2011-01-01 02:00:00,27,5,32,2011-01-01,2,5,1,1
3,2011-01-01 03:00:00,10,3,13,2011-01-01,3,5,1,1
4,2011-01-01 04:00:00,1,0,1,2011-01-01,4,5,1,1


## 6. Check result

In [76]:
hourly_rentals.info()
hourly_rentals.shape
hourly_rentals.head()
hourly_rentals.tail()

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   hour              17379 non-null  datetime64[us]
 1   registered_count  17379 non-null  int64         
 2   direct_count      17379 non-null  int64         
 3   total_count       17379 non-null  int64         
 4   date              17379 non-null  object        
 5   hour_of_day       17379 non-null  int32         
 6   day_of_week       17379 non-null  int32         
 7   month             17379 non-null  int32         
 8   is_weekend        17379 non-null  int64         
dtypes: datetime64[us](1), int32(3), int64(4), object(1)
memory usage: 1018.4+ KB


,hour,registered_count,direct_count,total_count,date,hour_of_day,day_of_week,month,is_weekend
17374,2012-12-31 19:00:00,108,11,119,2012-12-31,19,0,12,0
17375,2012-12-31 20:00:00,81,8,89,2012-12-31,20,0,12,0
17376,2012-12-31 21:00:00,83,7,90,2012-12-31,21,0,12,0
17377,2012-12-31 22:00:00,48,13,61,2012-12-31,22,0,12,0
17378,2012-12-31 23:00:00,37,12,49,2012-12-31,23,0,12,0


In [77]:
hourly_rentals.isna().sum()

hour                0
registered_count    0
direct_count        0
total_count         0
date                0
hour_of_day         0
day_of_week         0
month               0
is_weekend          0
dtype: int64

In [78]:
print("Hourly rental time range:")
print(hourly_rentals["hour"].min(), "to", hourly_rentals["hour"].max())

Hourly rental time range:
2011-01-01 00:00:00 to 2012-12-31 23:00:00


In [79]:
print("Original registered records:", len(registered))
print("Aggregated registered count:", hourly_rentals["registered_count"].sum())

print("\nOriginal direct records:", len(direct))
print("Aggregated direct count:", hourly_rentals["direct_count"].sum())

print("\nOriginal total records:", len(registered) + len(direct))
print("Aggregated total count:", hourly_rentals["total_count"].sum())

Original registered records: 2672662
Aggregated registered count: 2672662

Original direct records: 620017
Aggregated direct count: 620017

Original total records: 3292679
Aggregated total count: 3292679


## 7. Merge with weather data
### 7.1 Select features and create a clean weather table

In [89]:
weather = pd.read_csv("../data/weather.csv")
weather["datetime"] = pd.to_datetime(weather["datetime"])
# pd.api.types.is_datetime64_any_dtype(weather["datetime"])

In [81]:
# create same column "hour" for merging
weather["hour"] = weather["datetime"].dt.floor("h")
# weather[["datetime", "hour"]].head()

In [82]:
# remove unnecessary features
weather_features = weather.drop(columns=["id", "datetime"])
weather_features.head()

,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,hour
0,clear,3.3,3.0,81.0,0.0,2011-01-01 00:00:00
1,clear,2.3,2.0,80.0,0.0,2011-01-01 01:00:00
2,clear,2.3,2.0,80.0,0.0,2011-01-01 02:00:00
3,clear,3.3,3.0,75.0,0.0,2011-01-01 03:00:00
4,clear,3.3,3.0,75.0,0.0,2011-01-01 04:00:00


### 7.2 Merge with hourly_rentals

In [83]:
# left merge: hourly_rentals LEFT JOIN weather_features ON hour
rentals_with_weather = hourly_rentals.merge(
    weather_features,
    on="hour",
    how="left"
)

rentals_with_weather.head()

,hour,registered_count,direct_count,total_count,date,hour_of_day,day_of_week,month,is_weekend,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh
0,2011-01-01 00:00:00,13,3,16,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0
1,2011-01-01 01:00:00,32,8,40,2011-01-01,1,5,1,1,clear,2.3,2.0,80.0,0.0
2,2011-01-01 02:00:00,27,5,32,2011-01-01,2,5,1,1,clear,2.3,2.0,80.0,0.0
3,2011-01-01 03:00:00,10,3,13,2011-01-01,3,5,1,1,clear,3.3,3.0,75.0,0.0
4,2011-01-01 04:00:00,1,0,1,2011-01-01,4,5,1,1,clear,3.3,3.0,75.0,0.0


In [84]:
rentals_with_weather.info()
rentals_with_weather.shape

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   hour                     17379 non-null  datetime64[us]
 1   registered_count         17379 non-null  int64         
 2   direct_count             17379 non-null  int64         
 3   total_count              17379 non-null  int64         
 4   date                     17379 non-null  object        
 5   hour_of_day              17379 non-null  int32         
 6   day_of_week              17379 non-null  int32         
 7   month                    17379 non-null  int32         
 8   is_weekend               17379 non-null  int64         
 9   conditions               17379 non-null  str           
 10  temperature_c            17379 non-null  float64       
 11  perceived_temperature_c  17379 non-null  float64       
 12  humidity                 17379 non-null  fl

(17379, 14)

In [86]:
## missing value
rentals_with_weather[
    [
        "conditions",
        "temperature_c",
        "perceived_temperature_c",
        "humidity",
        "windspeed_kmh",
    ]
].isna().sum()

conditions                 0
temperature_c              0
perceived_temperature_c    0
humidity                   0
windspeed_kmh              0
dtype: int64

## 8. check result

In [87]:
# check row count
print("Before weather merge:", hourly_rentals.shape)
print("After weather merge:", rentals_with_weather.shape)

Before weather merge: (17379, 9)
After weather merge: (17379, 14)
